# Vie-GameEmo — Training (Simplified)

Notebook này gọi trực tiếp các scripts của project thay vì inline code.

**Pipeline:**
```
import_labels.py → stage0_preprocess.py → transcribe.py → extract_features.py → train.py
```

**Dataset layout cần có trước:**
```
data/
├── raw_videos/train/  ← clips .mp4
├── raw_videos/val/
├── raw_videos/test/
├── labels/train.json  ← [{id, video, choice, confidence}]
├── labels/val.json
└── labels/test.json
```


In [ ]:
# ============================================================
# CELL 1 — Môi trường
# ============================================================
import os, sys
WORKING = os.getcwd()
print(f'Working dir: {WORKING}')

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    print('⚠️  No GPU — training will be very slow')


In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Paths ---
DATASET_INPUT = '/kaggle/input/vie-gameemo-dataset'      # Kaggle input dataset
DATASET_LOCAL = os.path.join(WORKING, 'data')             # local fallback
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'          # project code

# --- Training ---
EPOCHS       = 30
BATCH_SIZE   = 16
FUSION_TYPE  = 'conv_attention_4m'
MIXED_PREC   = 'bf16'

# --- Visual strategy ---
# 'dual_path'  : face crops (webcam) + full-frame context (gameplay) — 4 modalities
# 'full_frame' : full-frame cho cả face lẫn context — 4 modalities, skip webcam detect
# 'face_only'  : chỉ face crops, context = zeros — 3 modalities
VISUAL_STRATEGY = 'dual_path'

# --- LLM stages (chạy sau perception) ---
TRAIN_LLM_PERCEPTION = False  # Stage 2a: align soft token → LLM predict nhãn (chỉ cần GT labels)
TRAIN_COGNITION      = False  # Stage 2b: joint recognition + reasoning (cần annotated descriptions)
TRAIN_RLVR           = False  # Stage 3:  RLVR reinforcement learning (optional)

In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
%pip install -q \
    "numpy<2" \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    faster-whisper>=1.0.3 \
    fasttext-wheel \
    scikit-learn \
    pydantic>=2.0 \
    librosa \
    torchvision \
    torchaudio \
    opencv-python-headless \
    tiktoken \
    sentencepiece

In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import shutil, subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
    print(f'Project: Kaggle input → {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/rhy221/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR], check=True)

# Pull latest changes
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    subprocess.run(['git', 'pull'], cwd=PROJECT_DIR, check=True)
    print('Project: pulled latest changes')

# Add to path
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Detect dataset
if os.path.exists(DATASET_INPUT):
    DATA_DIR = DATASET_INPUT
else:
    DATA_DIR = DATASET_LOCAL

SCRIPTS = os.path.join(PROJECT_DIR, 'scripts')
CONFIG  = os.path.join(PROJECT_DIR, 'config.yaml')
print(f'Data:    {DATA_DIR}')
print(f'Scripts: {SCRIPTS}')

## Bước 1 — Import labels + Tiền xử lý


In [ ]:
# ============================================================
# CELL 4b — Apply config overrides từ CELL 2
# ============================================================
import yaml

with open(CONFIG, encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Visual strategy
config['visual_encoder']['strategy'] = VISUAL_STRATEGY
if VISUAL_STRATEGY == 'face_only':
    config['fusion']['n_modalities'] = 3
else:
    config['fusion']['n_modalities'] = 4

with open(CONFIG, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'Config updated:')
print(f'  visual_encoder.strategy = {VISUAL_STRATEGY}')
print(f'  fusion.n_modalities     = {config["fusion"]["n_modalities"]}')

In [ ]:
# ============================================================
# CELL 6 — Tiền xử lý: tách audio + frames + webcam detect
# ============================================================
# MODE:
#   'full'        : audio + frames + webcam detect (lần đầu)
#   'webcam_only' : chỉ detect lại webcam (sau khi đổi backend/config)
PREPROCESS_MODE = 'webcam_only'

# Videos dir: tùy cấu trúc dataset
# Kaggle: videos nằm trong train/val/test trực tiếp dưới DATA_DIR
# Local:  videos nằm trong data/raw_videos/train/val/test
import os
_vdir = DATA_DIR if not os.path.exists(f'{DATA_DIR}/raw_videos') else f'{DATA_DIR}/raw_videos'

if PREPROCESS_MODE == 'webcam_only':
    !python {SCRIPTS}/stage0_preprocess.py \
        --config {CONFIG} \
        --videos-dir {_vdir} \
        --webcam-only
else:
    !python {SCRIPTS}/stage0_preprocess.py \
        --config {CONFIG} \
        --videos-dir {_vdir} \
        --resume

In [ ]:
# ============================================================
# CELL 7 — ASR: transcribe audio → cập nhật annotations
# ============================================================
# --overwrite : re-transcribe tất cả (dùng khi đổi config ASR/prompt)
OVERWRITE_TRANSCRIPT = True

_ow = '--overwrite' if OVERWRITE_TRANSCRIPT else ''
!python {SCRIPTS}/transcribe.py --config {CONFIG} {_ow}

In [ ]:
# ============================================================
# CELL 7b — Kiểm tra hallucination trong transcript
# ============================================================
import json
from pathlib import Path
from collections import Counter

annot_dir = Path('data/annotations')
HALLUC_PATTERNS = [
    'subscribe', 'like', 'kênh', 'lalaschool', 'đăng ký',
    'chia sẻ', 'bấm chuông', 'notification', 'bell',
    'comment', 'bình luận', 'video này',
]

suspect = []
lang_dist = Counter()
empty_count = 0
total = 0

for p in sorted(annot_dir.glob('*.json')):
    data = json.loads(p.read_text(encoding='utf-8'))
    transcript = data.get('transcript', '')
    lang = data.get('asr_detected_language', '?')
    lang_dist[lang] += 1
    total += 1

    if not transcript.strip():
        empty_count += 1
        continue

    lower = transcript.lower()
    matched = [pat for pat in HALLUC_PATTERNS if pat in lower]
    if matched:
        suspect.append({
            'clip': p.stem,
            'transcript': transcript[:100],
            'patterns': matched,
            'lang': lang,
        })

print(f'Total: {total} clips')
print(f'Empty transcript: {empty_count}')
print(f'Language distribution: {dict(lang_dist)}')
print(f'\nSuspect hallucination: {len(suspect)} clips')
print('=' * 70)
for s in suspect[:20]:
    print(f"  {s['clip']} [{s['lang']}] {s['patterns']}")
    print(f"    \"{s['transcript']}\"")
if len(suspect) > 20:
    print(f'  ... và {len(suspect) - 20} clips nữa')

# ============================================================
# CELL 8 — Extract + cache features (4 modality encoders)
# ============================================================
# MODALITIES: chọn modality cần extract
#   ['audio', 'face', 'context', 'text'] = tất cả (lần đầu)
#   ['text'] = chỉ text (nhanh, sau khi re-transcribe)
# OVERWRITE_CACHE: True để ghi đè cache cũ
MODALITIES = ['text']          # đổi thành ['audio', 'face', 'context', 'text'] cho lần đầu
OVERWRITE_CACHE = True         # True khi cần cập nhật text features sau re-transcribe

_ow_flag = '--overwrite' if OVERWRITE_CACHE else ''
_mod_flag = '--modalities ' + ' '.join(MODALITIES)
!python {SCRIPTS}/extract_features.py --config {CONFIG} {_mod_flag} {_ow_flag}

In [ ]:
# ============================================================
# CELL 8 — Extract + cache features (4 modality encoders)
# ============================================================
# --overwrite : ghi đè cache cũ (dùng khi đổi config encoder/strategy)
# Bỏ --overwrite để skip clips đã có cache (nhanh hơn khi chạy lại)
OVERWRITE_CACHE = False

_ow_flag = '--overwrite' if OVERWRITE_CACHE else ''
!python {SCRIPTS}/extract_features.py --config {CONFIG} {_ow_flag}

# Giải phóng VRAM + RAM sau khi extract xong


## Bước 3 — Training


In [ ]:
# ============================================================
# CELL 9 — Stage 1: Perception Training
# ============================================================
!python {SCRIPTS}/train.py \
    --config {CONFIG} \
    --stage perception \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --fusion {FUSION_TYPE}


In [ ]:
# ============================================================
# CELL 10 — Eval trên test split
# ============================================================
!python {SCRIPTS}/eval.py --config {CONFIG}


In [ ]:
# ============================================================
# CELL 10b — Phân tích kết quả + gợi ý tinh chỉnh
# ============================================================
# Đọc eval.json → phân tích per-class, confusion, rare class, language gap
# → in ra recommendations cụ thể.
!python {SCRIPTS}/analyze.py \
    --config {CONFIG} \
    --eval-json outputs/results/eval.json \
    --with-fragmentation


In [ ]:
# ============================================================
# CELL 10c — Demo LLM-1: Explainer (soft token + nhãn MLP → giải thích)
# ============================================================
# LLM-1 nhận nhãn từ MLP + soft token (nếu có ModalAdapter) → giải thích
# LLM-1 KHÔNG quyết định nhãn, chỉ giải thích tại sao nhãn đúng.
# Không cần train LLM — nhưng có ModalAdapter thì chất lượng tốt hơn.

import torch, json, gc
from pathlib import Path
from vie_gameemo.llm.llm1_explainer import LLM1Explainer
from vie_gameemo.data.schemas import EmotionLabel
from vie_gameemo.fusion import get_fusion
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.training.perception import load_checkpoint

N_DEMO = 5
LLM_DEMO_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LABEL_NAMES = [e.value for e in EmotionLabel]

best_ckpt = Path('outputs/checkpoints/perception_best.pt')
features_dir = Path('data/features')
annot_dir = Path('data/annotations')
splits_path = Path('data/splits.json')

# Tìm ModalAdapter checkpoint (ưu tiên llm_perception > cognition)
adapter_ckpt = None
for name in ['llm_perception_best.pt', 'cognition_best.pt']:
    p = Path('outputs/checkpoints') / name
    if p.exists():
        adapter_ckpt = str(p)
        break
if adapter_ckpt:
    print(f'ModalAdapter: {adapter_ckpt}')
else:
    print('Chưa có ModalAdapter — LLM-1 sẽ dùng text-only prompt (fallback)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load frozen perception model
fusion = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                    return_attention=False).to(device)
classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)

if best_ckpt.exists():
    load_checkpoint(best_ckpt, fusion, classifier)
    fusion.eval(); classifier.eval()
    print(f'Perception checkpoint loaded: {best_ckpt}')
else:
    print('Chưa có perception checkpoint — chạy CELL 9 trước')

# Load LLM-1
llm = LLM1Explainer(
    model_name=LLM_DEMO_MODEL,
    quantization='4bit',
    max_new_tokens=300,
    temperature=0.7,
    modal_adapter_ckpt=adapter_ckpt,
)
llm.load()

# Lấy test clips
with open(splits_path, encoding='utf-8') as f:
    splits = json.load(f)

test_clips = [cid for cid, s in splits.items() if s == 'test']
demo_clips = [cid for cid in test_clips
              if (annot_dir / f'{cid}.json').exists()
              and (features_dir / f'{cid}.pt').exists()][:N_DEMO]

print(f'\nDemo LLM-1 (Explainer) trên {len(demo_clips)} clips:\n')
print('=' * 70)

for cid in demo_clips:
    ann = json.loads((annot_dir / f'{cid}.json').read_text(encoding='utf-8'))
    features = torch.load(features_dir / f'{cid}.pt', map_location=device)

    with torch.no_grad():
        audio = features['audio'].unsqueeze(0).to(device)
        face = features['face'].unsqueeze(0).to(device)
        context = features['context'].unsqueeze(0).to(device)
        text = features['text'].unsqueeze(0).to(device)
        fused = fusion(audio, face, context, text)
        if isinstance(fused, tuple):
            fused = fused[0]
        logits = classifier(fused)
        pred_idx = int(logits.argmax(dim=-1).item())
        mlp_label = LABEL_NAMES[pred_idx]
        confidence = float(torch.softmax(logits, dim=-1)[0, pred_idx].item())

    result = llm.reason({
        'label': mlp_label,
        'fusion_emb': fused,
        'transcript': ann.get('transcript', ''),
        'source_language': ann.get('source_language', 'vi'),
    })

    print(f'Clip: {cid}')
    print(f'MLP: {mlp_label} ({confidence:.2f}) | GT: {ann.get("emotion_label", "?")}')
    print(f'Transcript: "{ann.get("transcript", "")[:80]}"')
    print(f'LLM-1 Reasoning: {result.reasoning}')
    print(f'  → Answer: {result.answer} (format_valid={result.format_valid})')
    print('-' * 70)

llm.unload()
del llm, fusion, classifier; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n✅ Demo LLM-1 xong.')

# ============================================================
# CELL 10c — Demo LLM-1: Explainer (soft token + nhãn MLP → giải thích)
# ============================================================
# LLM-1 nhận nhãn từ MLP + soft token (nếu có ModalAdapter) → giải thích
# Dùng 2 fusion riêng: fusion_mlp (cho MLP label) + fusion_llm (cho soft token)

import torch, json, gc
from pathlib import Path
from vie_gameemo.llm.llm1_explainer import LLM1Explainer
from vie_gameemo.data.schemas import EmotionLabel
from vie_gameemo.fusion import get_fusion
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.training.perception import load_checkpoint

N_DEMO = 5
LLM_DEMO_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LABEL_NAMES = [e.value for e in EmotionLabel]

perception_ckpt = Path('outputs/checkpoints/perception_best.pt')
llm_ckpt = Path('outputs/checkpoints/llm_perception_best.pt')
features_dir = Path('data/features')
annot_dir = Path('data/annotations')
splits_path = Path('data/splits.json')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Fusion for MLP (predict label) ---
fusion_mlp = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                         return_attention=False).to(device)
classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
if perception_ckpt.exists():
    load_checkpoint(perception_ckpt, fusion_mlp, classifier)
    fusion_mlp.eval(); classifier.eval()
    print(f'MLP fusion loaded: {perception_ckpt}')
else:
    print('Chua co perception checkpoint — chay CELL 9 truoc')

# --- Fusion for LLM (soft token) ---
adapter_ckpt = None
fusion_llm = None
if llm_ckpt.exists():
    adapter_ckpt = str(llm_ckpt)
    fusion_llm = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                             return_attention=False).to(device)
    llm_state = torch.load(llm_ckpt, map_location='cpu')
    if 'fusion_state_dict' in llm_state:
        fusion_llm.load_state_dict(llm_state['fusion_state_dict'])
        fusion_llm.eval()
        print(f'LLM fusion loaded: {llm_ckpt}')
    else:
        fusion_llm = fusion_mlp
        print('LLM checkpoint has no separate fusion — using MLP fusion')
else:
    print('Chua co LLM perception checkpoint — LLM-1 dung text-only fallback')

# Load LLM-1
llm = LLM1Explainer(
    model_name=LLM_DEMO_MODEL,
    quantization='4bit',
    max_new_tokens=300,
    temperature=0.7,
    modal_adapter_ckpt=adapter_ckpt,
)
llm.load()

# Demo
with open(splits_path, encoding='utf-8') as f:
    splits = json.load(f)

test_clips = [cid for cid, s in splits.items() if s == 'test']
demo_clips = [cid for cid in test_clips
              if (annot_dir / f'{cid}.json').exists()
              and (features_dir / f'{cid}.pt').exists()][:N_DEMO]

print(f'\nDemo LLM-1 (Explainer) tren {len(demo_clips)} clips:\n')
print('=' * 70)

for cid in demo_clips:
    ann = json.loads((annot_dir / f'{cid}.json').read_text(encoding='utf-8'))
    features = torch.load(features_dir / f'{cid}.pt', map_location=device)

    audio = features['audio'].unsqueeze(0).to(device)
    face = features['face'].unsqueeze(0).to(device)
    context = features['context'].unsqueeze(0).to(device)
    text = features['text'].unsqueeze(0).to(device)

    with torch.no_grad():
        # MLP fusion → label
        fused_mlp = fusion_mlp(audio, face, context, text)
        if isinstance(fused_mlp, tuple):
            fused_mlp = fused_mlp[0]
        logits = classifier(fused_mlp)
        pred_idx = int(logits.argmax(dim=-1).item())
        mlp_label = LABEL_NAMES[pred_idx]
        confidence = float(torch.softmax(logits, dim=-1)[0, pred_idx].item())

        # LLM fusion → soft token
        use_fusion = fusion_llm if fusion_llm is not None else fusion_mlp
        fused_llm = use_fusion(audio, face, context, text)
        if isinstance(fused_llm, tuple):
            fused_llm = fused_llm[0]

    result = llm.reason({
        'label': mlp_label,
        'fusion_emb': fused_llm,
        'transcript': ann.get('transcript', ''),
        'source_language': ann.get('source_language', 'vi'),
    })

    print(f'Clip: {cid}')
    print(f'MLP: {mlp_label} ({confidence:.2f}) | GT: {ann.get("emotion_label", "?")}')
    print(f'Transcript: "{ann.get("transcript", "")[:80]}"')
    print(f'LLM-1 Reasoning: {result.reasoning}')
    print(f'  -> Answer: {result.answer} (format_valid={result.format_valid})')
    print('-' * 70)

llm.unload()
del llm, fusion_mlp, fusion_llm, classifier; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\nDemo LLM-1 xong.')

In [ ]:
# ============================================================
# CELL 11 — Stage 2a: LLM Perception (align soft token → predict nhãn)
# ============================================================
# Chỉ cần GT labels, KHÔNG cần annotated descriptions.
# Train ModalAdapter + LLM LoRA để LLM hiểu soft token → predict nhãn.
# active_setup trong config quyết định mode:
#   llm1/llm3: soft token only → predict nhãn
#   llm2:      soft token + MLP label hint → predict nhãn
if TRAIN_LLM_PERCEPTION:
    !python {SCRIPTS}/train.py \
        --config {CONFIG} \
        --stage llm_perception \
        --resume-from outputs/checkpoints/perception_best.pt
else:
    print('TRAIN_LLM_PERCEPTION=False — bỏ qua. Đổi ở CELL 2 để bật.')

In [ ]:
# ============================================================
# CELL 11a — Stage 2b: Cognition (optional, cần annotated descriptions)
# ============================================================
# Củng cố reasoning: LLM learn generate giải thích mạch lạc.
# Cần chạy Stage 0 annotation pipeline (notebook 01) trước để có
# reasoning text targets.
if TRAIN_COGNITION:
    !python {SCRIPTS}/train.py \
        --config {CONFIG} \
        --stage cognition \
        --resume-from outputs/checkpoints/perception_best.pt
else:
    print('TRAIN_COGNITION=False — bỏ qua. Cần annotated descriptions để bật.')

In [ ]:
# ============================================================
# CELL 11c — Demo LLM-3: Pure Reasoner (chỉ soft token → tự dự đoán)
# ============================================================
# LLM-3 KHÔNG nhận nhãn MLP — chỉ dùng soft token.
# So sánh kết quả LLM-3 vs MLP để đánh giá LLM có học được từ fusion không.
# Yêu cầu: đã chạy cognition training (CELL 11).

if TRAIN_COGNITION:
    import torch, json, gc
    from pathlib import Path
    from vie_gameemo.llm.llm3_vlm import LLM3PureReasoner
    from vie_gameemo.data.schemas import EmotionLabel
    from vie_gameemo.fusion import get_fusion
    from vie_gameemo.classifiers.mlp import EmotionClassifier
    from vie_gameemo.training.perception import load_checkpoint

    N_DEMO = 5
    LABEL_NAMES = [e.value for e in EmotionLabel]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cognition_ckpt = Path('outputs/checkpoints/cognition_best.pt')

    # Load perception model (chỉ để so sánh MLP vs LLM-3)
    fusion = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                        return_attention=False).to(device)
    classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
    load_checkpoint(Path('outputs/checkpoints/perception_best.pt'), fusion, classifier)
    fusion.eval(); classifier.eval()

    # Load LLM-3
    llm3 = LLM3PureReasoner(
        model_name='Qwen/Qwen2.5-7B-Instruct',
        quantization='4bit',
        modal_adapter_ckpt=cognition_ckpt,
    )
    llm3.load()

    # Demo
    features_dir = Path('data/features')
    annot_dir = Path('data/annotations')
    with open(Path('data/splits.json'), encoding='utf-8') as f:
        splits = json.load(f)

    test_clips = [cid for cid, s in splits.items() if s == 'test']
    demo_clips = [cid for cid in test_clips
                  if (annot_dir / f'{cid}.json').exists()
                  and (features_dir / f'{cid}.pt').exists()][:N_DEMO]

    print(f'Demo LLM-3 (Pure Reasoner) trên {len(demo_clips)} clips:\n')
    print('=' * 70)
    mlp_correct = 0
    llm3_correct = 0

    for cid in demo_clips:
        ann = json.loads((annot_dir / f'{cid}.json').read_text(encoding='utf-8'))
        gt = ann.get('emotion_label', 'neutral')
        features = torch.load(features_dir / f'{cid}.pt', map_location=device)

        with torch.no_grad():
            audio = features['audio'].unsqueeze(0).to(device)
            face = features['face'].unsqueeze(0).to(device)
            context = features['context'].unsqueeze(0).to(device)
            text = features['text'].unsqueeze(0).to(device)
            fused = fusion(audio, face, context, text)
            if isinstance(fused, tuple):
                fused = fused[0]
            logits = classifier(fused)
            pred_idx = int(logits.argmax(dim=-1).item())
            mlp_label = LABEL_NAMES[pred_idx]

        result = llm3.reason({'fusion_emb': fused})

        if mlp_label == gt:
            mlp_correct += 1
        if result.answer == gt:
            llm3_correct += 1

        print(f'Clip: {cid} | GT: {gt}')
        print(f'  MLP:   {mlp_label} {"✓" if mlp_label == gt else "✗"}')
        print(f'  LLM-3: {result.answer} {"✓" if result.answer == gt else "✗"}')
        print(f'  Reasoning: {result.reasoning[:150]}')
        print('-' * 70)

    print(f'\nAccuracy: MLP={mlp_correct}/{len(demo_clips)}, LLM-3={llm3_correct}/{len(demo_clips)}')

    llm3.unload()
    del llm3, fusion, classifier; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('TRAIN_COGNITION=False — cần train cognition trước để dùng LLM-3.')

In [ ]:
# ============================================================
# CELL 11c — Demo LLM-3: Pure Reasoner (chỉ soft token → tự dự đoán)
# ============================================================
# Dùng fusion_mlp cho MLP (so sánh), fusion_llm cho LLM-3 soft token.
# Yêu cầu: đã chạy LLM perception training (CELL 11).

if TRAIN_LLM_PERCEPTION or Path('outputs/checkpoints/llm_perception_best.pt').exists():
    import torch, json, gc
    from pathlib import Path
    from vie_gameemo.llm.llm3_vlm import LLM3PureReasoner
    from vie_gameemo.data.schemas import EmotionLabel
    from vie_gameemo.fusion import get_fusion
    from vie_gameemo.classifiers.mlp import EmotionClassifier
    from vie_gameemo.training.perception import load_checkpoint

    N_DEMO = 5
    LABEL_NAMES = [e.value for e in EmotionLabel]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    llm_ckpt = Path('outputs/checkpoints/llm_perception_best.pt')

    # Fusion MLP (so sánh)
    fusion_mlp = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                             return_attention=False).to(device)
    classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
    load_checkpoint(Path('outputs/checkpoints/perception_best.pt'), fusion_mlp, classifier)
    fusion_mlp.eval(); classifier.eval()

    # Fusion LLM
    fusion_llm = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                             return_attention=False).to(device)
    llm_state = torch.load(llm_ckpt, map_location='cpu')
    if 'fusion_state_dict' in llm_state:
        fusion_llm.load_state_dict(llm_state['fusion_state_dict'])
    else:
        fusion_llm.load_state_dict(fusion_mlp.state_dict())
    fusion_llm.eval()

    llm3 = LLM3PureReasoner(
        model_name='Qwen/Qwen2.5-7B-Instruct',
        quantization='4bit',
        modal_adapter_ckpt=llm_ckpt,
    )
    llm3.load()

    features_dir = Path('data/features')
    annot_dir = Path('data/annotations')
    with open(Path('data/splits.json'), encoding='utf-8') as f:
        splits = json.load(f)

    test_clips = [cid for cid, s in splits.items() if s == 'test']
    demo_clips = [cid for cid in test_clips
                  if (annot_dir / f'{cid}.json').exists()
                  and (features_dir / f'{cid}.pt').exists()][:N_DEMO]

    print(f'Demo LLM-3 (Pure Reasoner) tren {len(demo_clips)} clips:\n')
    print('=' * 70)
    mlp_correct = 0
    llm3_correct = 0

    for cid in demo_clips:
        ann = json.loads((annot_dir / f'{cid}.json').read_text(encoding='utf-8'))
        gt = ann.get('emotion_label', 'neutral')
        features = torch.load(features_dir / f'{cid}.pt', map_location=device)

        audio = features['audio'].unsqueeze(0).to(device)
        face = features['face'].unsqueeze(0).to(device)
        context = features['context'].unsqueeze(0).to(device)
        text = features['text'].unsqueeze(0).to(device)

        with torch.no_grad():
            fused_mlp = fusion_mlp(audio, face, context, text)
            if isinstance(fused_mlp, tuple): fused_mlp = fused_mlp[0]
            logits = classifier(fused_mlp)
            mlp_label = LABEL_NAMES[int(logits.argmax(dim=-1).item())]

            fused_llm = fusion_llm(audio, face, context, text)
            if isinstance(fused_llm, tuple): fused_llm = fused_llm[0]

        result = llm3.reason({'fusion_emb': fused_llm})

        if mlp_label == gt: mlp_correct += 1
        if result.answer == gt: llm3_correct += 1

        print(f'Clip: {cid} | GT: {gt}')
        print(f'  MLP:   {mlp_label} {"V" if mlp_label == gt else "X"}')
        print(f'  LLM-3: {result.answer} {"V" if result.answer == gt else "X"}')
        print(f'  Reasoning: {result.reasoning[:150]}')
        print('-' * 70)

    print(f'\nAccuracy: MLP={mlp_correct}/{len(demo_clips)}, LLM-3={llm3_correct}/{len(demo_clips)}')

    llm3.unload()
    del llm3, fusion_mlp, fusion_llm, classifier; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('Chua co llm_perception_best.pt — chay CELL 11 truoc.')

In [ ]:
# ============================================================
# CELL 11b — Demo LLM-2: Co-Reasoner (soft token + nhãn MLP hint)
# ============================================================
# Dùng fusion_mlp cho MLP label, fusion_llm cho soft token.
# Yêu cầu: đã chạy LLM perception training (CELL 11).

if TRAIN_LLM_PERCEPTION or Path('outputs/checkpoints/llm_perception_best.pt').exists():
    import torch, json, gc
    from pathlib import Path
    from vie_gameemo.llm.llm2_coreasoner import LLM2CoReasoner
    from vie_gameemo.data.schemas import EmotionLabel
    from vie_gameemo.fusion import get_fusion
    from vie_gameemo.classifiers.mlp import EmotionClassifier
    from vie_gameemo.training.perception import load_checkpoint

    N_DEMO = 5
    LABEL_NAMES = [e.value for e in EmotionLabel]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    llm_ckpt = Path('outputs/checkpoints/llm_perception_best.pt')

    # Fusion MLP
    fusion_mlp = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                             return_attention=False).to(device)
    classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
    load_checkpoint(Path('outputs/checkpoints/perception_best.pt'), fusion_mlp, classifier)
    fusion_mlp.eval(); classifier.eval()

    # Fusion LLM
    fusion_llm = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                             return_attention=False).to(device)
    llm_state = torch.load(llm_ckpt, map_location='cpu')
    if 'fusion_state_dict' in llm_state:
        fusion_llm.load_state_dict(llm_state['fusion_state_dict'])
    else:
        fusion_llm.load_state_dict(fusion_mlp.state_dict())
    fusion_llm.eval()

    llm2 = LLM2CoReasoner(
        model_name='Qwen/Qwen2.5-7B-Instruct',
        quantization='4bit',
        modal_adapter_ckpt=llm_ckpt,
    )
    llm2.load()

    features_dir = Path('data/features')
    annot_dir = Path('data/annotations')
    with open(Path('data/splits.json'), encoding='utf-8') as f:
        splits = json.load(f)

    test_clips = [cid for cid, s in splits.items() if s == 'test']
    demo_clips = [cid for cid in test_clips
                  if (annot_dir / f'{cid}.json').exists()
                  and (features_dir / f'{cid}.pt').exists()][:N_DEMO]

    print(f'Demo LLM-2 (Co-Reasoner) tren {len(demo_clips)} clips:\n')
    print('=' * 70)
    agree_count = 0

    for cid in demo_clips:
        ann = json.loads((annot_dir / f'{cid}.json').read_text(encoding='utf-8'))
        features = torch.load(features_dir / f'{cid}.pt', map_location=device)

        audio = features['audio'].unsqueeze(0).to(device)
        face = features['face'].unsqueeze(0).to(device)
        context = features['context'].unsqueeze(0).to(device)
        text = features['text'].unsqueeze(0).to(device)

        with torch.no_grad():
            fused_mlp = fusion_mlp(audio, face, context, text)
            if isinstance(fused_mlp, tuple): fused_mlp = fused_mlp[0]
            logits = classifier(fused_mlp)
            mlp_label = LABEL_NAMES[int(logits.argmax(dim=-1).item())]

            fused_llm = fusion_llm(audio, face, context, text)
            if isinstance(fused_llm, tuple): fused_llm = fused_llm[0]

        result = llm2.reason({
            'fusion_emb': fused_llm,
            'mlp_label': mlp_label,
            'transcript': ann.get('transcript', ''),
            'source_language': ann.get('source_language', 'vi'),
        })

        agreed = result.answer == mlp_label
        if agreed: agree_count += 1
        status = 'dong y' if agreed else f'override -> {result.answer}'

        print(f'Clip: {cid}')
        print(f'MLP: {mlp_label} | LLM-2: {status} | GT: {ann.get("emotion_label", "?")}')
        print(f'Reasoning: {result.reasoning[:150]}')
        print('-' * 70)

    print(f'\nLLM-2 dong y voi MLP: {agree_count}/{len(demo_clips)}')

    llm2.unload()
    del llm2, fusion_mlp, fusion_llm, classifier; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('Chua co llm_perception_best.pt — chay CELL 11 truoc.')

In [ ]:
# ============================================================
# CELL 13 — Lưu checkpoint để download
# ============================================================
import zipfile
from pathlib import Path

CKPT_DIR = 'outputs/checkpoints'
ckpt_files = list(Path(CKPT_DIR).glob('*.pt'))
print(f'Checkpoints ({len(ckpt_files)}):')
for f in ckpt_files:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

archive = os.path.join(WORKING, 'vie_gameemo_checkpoints.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ckpt in ckpt_files:
        zf.write(str(ckpt), f'checkpoints/{ckpt.name}')
    zf.write(CONFIG, 'config.yaml')

print(f'\n✅ Archive: {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')


In [ ]:
# ============================================================
# CELL TEST — End-to-end: MLP phan loai + LLM-1 giai thich
# ============================================================
# Yeu cau:
#   1. outputs/checkpoints/perception_best.pt        (Stage 1)
#   2. outputs/checkpoints/llm1_explanation_best.pt  (sau train_llm1.py --stage a)
# Neu chua co llm1_explanation_best.pt -> LLM-1 fallback text-only prompt.

# Compat shim: torch.utils.serialization removed in newer PyTorch
import sys, types
sys.modules.setdefault('torch.utils.serialization', types.ModuleType('torch.utils.serialization'))

import torch, json, gc, random
import torch.nn as nn
from pathlib import Path
from vie_gameemo.llm.llm1_explainer import LLM1Explainer
from vie_gameemo.data.schemas import EmotionLabel
from vie_gameemo.fusion import get_fusion
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.training.perception import load_checkpoint, infer_fusion_dims_from_checkpoint
from vie_gameemo.training.llm1_explanation import GHeadPerModality

# ---- Config -------------------------------------------------------
N_DEMO          = 5
LLM_MODEL       = 'Qwen/Qwen2.5-7B-Instruct'
LABEL_NAMES     = [e.value for e in EmotionLabel]
PERCEPTION_CKPT = Path('outputs/checkpoints/perception_best.pt')
LLM1_CKPT       = Path('outputs/checkpoints/llm1_explanation_best.pt')
FEATURES_DIR    = Path('data/features')
ANNOT_DIR       = Path('data/annotations')
SPLITS_PATH     = Path('data/splits.json')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Load Perception (Fusion + MLP) -------------------------------
if not PERCEPTION_CKPT.exists():
    raise FileNotFoundError(f'Thieu {PERCEPTION_CKPT}')

extra_dims = infer_fusion_dims_from_checkpoint(PERCEPTION_CKPT, d_model=768)
print(f'Checkpoint dims: {extra_dims}')

fusion = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                    return_attention=False, **extra_dims).to(device)
classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
load_checkpoint(PERCEPTION_CKPT, fusion, classifier)
fusion.eval(); classifier.eval()
print(f'Perception: loaded ({PERCEPTION_CKPT.name})')

# ---- Load LLM-1 + GHead from checkpoint --------------------------
adapter_ckpt = str(LLM1_CKPT) if LLM1_CKPT.exists() else None
g_head = None

if LLM1_CKPT.exists():
    _ckpt = torch.load(LLM1_CKPT, map_location='cpu', weights_only=False)
    if 'g_head' in _ckpt:
        _gh_sd = _ckpt['g_head']
        # Infer dims from saved weights
        _face_dim    = _gh_sd['face_head.0.weight'].shape[1]
        _audio_dim   = _gh_sd['voice_head.0.weight'].shape[1]
        _text_dim_gh = _gh_sd['text_head.0.weight'].shape[1]
        _has_ctx     = 'motion_head.0.weight' in _gh_sd
        _ctx_dim     = _gh_sd['motion_head.0.weight'].shape[1] if _has_ctx else 768
        g_head = GHeadPerModality(
            face_dim=_face_dim, audio_dim=_audio_dim,
            text_dim=_text_dim_gh, context_dim=_ctx_dim,
            has_context=_has_ctx,
        ).to(device)
        g_head.load_state_dict(_gh_sd)
        g_head.eval()
        print(f'GHead loaded (face_dim={_face_dim}, audio_dim={_audio_dim})')
    else:
        print('GHead not in checkpoint — skip grounded cues')

print(f'LLM-1 checkpoint: {LLM1_CKPT.name if adapter_ckpt else "khong co — fallback text-only"}')
llm1 = LLM1Explainer(
    model_name=LLM_MODEL, quantization='4bit',
    max_new_tokens=200, temperature=0.7,
    modal_adapter_ckpt=adapter_ckpt,
)
llm1.load()
print(f'LLM-1 loaded | trained_mode={llm1._trained_mode}\n')


# ---- Grounded cue conversion from GHead outputs ------------------
def _bin(val, bins):
    for thr, label in bins:
        if val < thr:
            return label
    return bins[-1][1]

# attr_vec scale (from cue_extractor.py):
#   face: [ear(raw), mar(raw), brow_h(raw), yaw/45, pitch/45]
#   voice: [f0/300, rms/0.1, rate/5]
_FACE_EAR  = [(0.18, 'mat-he'), (0.25, 'mat-binh-thuong'), (float('inf'), 'mat-mo')]
_FACE_MAR  = [(0.15, 'mieng-khep'), (0.40, 'mieng-he'), (float('inf'), 'mieng-mo')]
_FACE_BROW = [(0.03, 'long-may-thap'), (0.06, 'long-may-binh-thuong'), (float('inf'), 'long-may-cau')]
_FACE_YAW  = [(-0.33, 'mat-quay-trai'), (0.33, 'mat-nhin-thang'), (float('inf'), 'mat-quay-phai')]
_FACE_PITCH= [(-0.22, 'dau-cuoi'), (0.22, 'dau-thang'), (float('inf'), 'dau-ngoc')]
_V_F0      = [(0.50, 'giong-tram'), (0.83, 'giong-trung'), (float('inf'), 'giong-cao')]
_V_RMS     = [(0.20, 'noi-nho'), (0.60, 'noi-vua'), (float('inf'), 'noi-to')]
_V_RATE    = [(0.40, 'noi-cham'), (0.80, 'toc-do-binh-thuong'), (float('inf'), 'noi-nhanh')]


def extract_grounded_cues(g_head, face, audio, context, text, has_face):
    """Run GHead on raw (B,T,D) embeddings -> grounded cue string.
    GHead.forward already does mean-pool internally.
    """
    with torch.no_grad():
        face_pred, voice_pred, motion_pred, text_pred = g_head(
            face, audio, context, text
        )
    f = face_pred[0].cpu()
    v = voice_pred[0].cpu()

    face_detected = bool(has_face.any().item())
    if face_detected:
        ear, mar, brow, yaw, pitch = f[0].item(), f[1].item(), f[2].item(), f[3].item(), f[4].item()
        face_str = (
            f"{_bin(ear, _FACE_EAR)}, {_bin(mar, _FACE_MAR)}, "
            f"{_bin(brow, _FACE_BROW)}, {_bin(yaw, _FACE_YAW)}, {_bin(pitch, _FACE_PITCH)}"
        )
    else:
        face_str = 'khong-phat-hien-khuon-mat'

    f0, rms, rate = v[0].item(), v[1].item(), v[2].item()
    voice_str = f"{_bin(f0, _V_F0)}, {_bin(rms, _V_RMS)}, {_bin(rate, _V_RATE)}"

    return face_str, voice_str


def verbalize(llm, face_str, voice_str, transcript, label):
    """Convert grounded cues + transcript to natural language explanation."""
    parts = [f'Khuon mat: {face_str}.', f'Giong noi: {voice_str}.']
    if transcript:
        parts.append(f'Loi noi: "{transcript[:120]}".')
    parts.append(
        f'Cam xuc duoc phan loai: {label}. '
        f'Viet 1-2 cau tieng Viet giai thich ngan gon tai sao streamer the hien trang thai {label}. '
        f'Chi viet cau giai thich bang tieng Viet, khong giai thich them.'
    )
    prompt = '\n'.join(parts)
    msgs = [{'role': 'user', 'content': prompt}]
    text_input = llm.tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = llm.tokenizer(text_input, return_tensors='pt').to(llm.model.device)
    with torch.no_grad():
        out = llm.model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=llm.tokenizer.eos_token_id,
        )
    return llm.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ---- Lay test clips ngau nhien ------------------------------------
with open(SPLITS_PATH, encoding='utf-8') as f:
    splits = json.load(f)

_all_test = [
    cid for cid, s in splits.items()
    if s == 'test'
    and (ANNOT_DIR / f'{cid}.json').exists()
    and (FEATURES_DIR / f'{cid}.pt').exists()
]
test_clips = random.sample(_all_test, min(N_DEMO, len(_all_test)))

print(f'Test clips: {len(test_clips)}')
print('=' * 70)

n_mlp_correct = 0
n_fmt_valid   = 0

for cid in test_clips:
    ann = json.loads((ANNOT_DIR / f'{cid}.json').read_text(encoding='utf-8'))
    gt  = ann.get('emotion_label', '?')
    feat = torch.load(FEATURES_DIR / f'{cid}.pt', map_location=device, weights_only=False)

    audio   = feat['audio'].unsqueeze(0).to(device)
    face    = feat['face'].unsqueeze(0).to(device)
    context = feat['context'].unsqueeze(0).to(device)
    text    = feat['text'].unsqueeze(0).to(device)
    raw_hf  = feat.get('has_face', torch.tensor(True))
    if raw_hf.dim() == 0:
        raw_hf = raw_hf.unsqueeze(0)
    has_face = raw_hf.to(device)

    with torch.no_grad():
        fused = fusion(audio, face, context, text)
        if isinstance(fused, tuple):
            fused = fused[0]
        logits, penult = classifier(fused, return_penultimate=True)
        pred_idx   = int(logits.argmax(dim=-1).item())
        mlp_label  = LABEL_NAMES[pred_idx]
        confidence = float(torch.softmax(logits, dim=-1)[0, pred_idx].item())

    result = llm1.reason({
        'label': mlp_label, 'fusion_emb': fused, 'penult': penult,
        'audio_emb': audio, 'face_emb': face, 'context_emb': context,
        'has_face': has_face,
        'transcript': ann.get('transcript', ''),
        'source_language': ann.get('source_language', 'vi'),
    })

    # Grounded cues from GHead (if available)
    transcript = ann.get('transcript', '')
    if g_head is not None:
        face_str, voice_str = extract_grounded_cues(g_head, face, audio, context, text, has_face)
        explanation = verbalize(llm1, face_str, voice_str, transcript, result.answer)
    else:
        face_str = voice_str = 'N/A'
        explanation = result.reasoning

    mlp_ok = (mlp_label == gt)
    n_mlp_correct += mlp_ok
    n_fmt_valid   += result.format_valid

    print(f'Clip       : {cid}')
    print(f'GT         : {gt}')
    print(f'MLP        : {mlp_label} ({confidence:.0%})  {"OK" if mlp_ok else "WRONG"}')
    print(f'Transcript : "{transcript[:80]}"')
    print(f'Face cues  : {face_str}')
    print(f'Voice cues : {voice_str}')
    print(f'Cues (LLM) : {result.reasoning[:120]}')
    print(f'Giai thich : {explanation}')
    print(f'Emotion    : {result.answer}  (format_valid={result.format_valid})')
    print('-' * 70)

print()
print(f'MLP accuracy : {n_mlp_correct}/{len(test_clips)}')
print(f'Format valid : {n_fmt_valid}/{len(test_clips)}')

llm1.unload()
del llm1, fusion, classifier
if g_head is not None:
    del g_head
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Done.')